# Lesson 9.1 · Boosting

Random forest reduces the variance by **averaging** many deep trees in parallel. **Boosting** does the opposite: it builds **small, weak trees, one after the other**, and each tree corrects the errors of the previous ones. A sum of many small steps in the right direction:
it is the algorithm that wins most competitions on tabular data, and it is the one behind XGBoost, LightGBM and CatBoost.

**What you will learn**

- The idea of gradient boosting: fit each tree to the **residuals** of the current model.
- The role of **shrinkage** (`learning_rate`) and of the rounds.
- Why boosting can overfit (unlike the forest) and how to see it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
X = rng.uniform(-3, 3, (600, 1))
y = np.sin(X[:, 0]) + 0.3 * X[:, 0] + rng.normal(0, 0.3, 600)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)

## Trees on the residuals

With the squared loss, boosting is very simple:

1. Start from a constant prediction (the mean of `y`): $F_0$.
2. Compute the **residuals** $r = y - F_{m-1}(x)$: what the current model gets wrong.
3. Train a **small** tree (depth 2-6) to predict the residuals.
4. Update $F_m = F_{m-1} + \eta\,h_m(x)$, where $\eta$ is the *learning rate* (shrinkage, typically 0.01-0.3).
5. Repeat.

For other losses (logistic, quantile) the "residuals" become the **negative gradient** of the loss with respect to the prediction, hence the name *gradient* boosting.

In [ ]:
eta, rounds = 0.3, 5
F = np.full(len(y_tr), y_tr.mean())                         # F0: the mean
grid = np.linspace(-3, 3, 200).reshape(-1, 1)
Fg = np.full(len(grid), y_tr.mean())
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for m in range(1, rounds + 1):
    tree = DecisionTreeRegressor(max_depth=2).fit(X_tr, y_tr - F)          # SMALL tree on the residuals
    F += eta * tree.predict(X_tr); Fg += eta * tree.predict(grid)
    if m in (1, 3, 5):
        a = ax[[1, 3, 5].index(m)]
        a.scatter(X_tr, y_tr, s=5, color="gray", alpha=0.5); a.plot(grid, Fg, "r"); a.set_title(f"after {m} trees | train error {np.mean((y_tr - F) ** 2):.3f}")
plt.tight_layout(); plt.show()

Each tree alone is very little (2 levels, 4 leaves), but the sum of a few steps already follows the shape of the relationship. It is the opposite of the forest: **high bias, low variance per tree**, and the complexity grows at every round.

## Shrinkage and rounds: a trade-off

A small `learning_rate` takes small steps: you need **more rounds** but the generalization is better (each tree weighs little and cannot "memorize"). With a large `learning_rate` a few rounds are enough and the risk of overfitting is high. That is why `learning_rate` and `n_estimators` have to be chosen
together: a practical rule is a low `learning_rate` (0.03-0.1) and the number of rounds chosen with **early stopping** (lesson 9.2).

In [ ]:
for eta in [0.5, 0.1, 0.02]:
    gb = GradientBoostingRegressor(n_estimators=400, learning_rate=eta, max_depth=2, random_state=0).fit(X_tr, y_tr)
    curve = [mean_squared_error(y_te, p) for p in gb.staged_predict(X_te)]           # test error after each round
    print(f"learning_rate {eta:4}: minimum test error {min(curve):.4f} at round {int(np.argmin(curve)) + 1:3d} | after 400 rounds {curve[-1]:.4f}")

With `learning_rate = 0.5` the minimum arrives immediately and then the error **climbs back** (overfitting); with 0.02 the curve drops slowly and after 400 rounds it is still close to the minimum. **Unlike the forest, boosting overfits as the rounds increase**: you have to stop.

> **Common mistake.** Fixing `n_estimators=1000` "to be safe" without early stopping: with a medium learning rate and a noisy dataset you are just memorizing the training set.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why are small trees used in boosting?</summary>

Because each tree must be a weak corrector: the sum of many small steps in directions guided by the gradient builds a flexible function in a controlled way, with the bias dropping at every round. Deep trees would memorize the residuals already at the first step.

</details>

<details><summary>What is the difference between bagging and boosting?</summary>

Bagging trains many deep trees **in parallel** on different samples and averages them (it reduces the variance). Boosting trains small trees **in sequence**, each one on the errors of the previous one (it reduces the bias, and with too many rounds it increases the variance).

</details>